# **Moneyball**

This dataset contains information about players, teams, performance, and salaries in Major League Baseball (MLB). It enables analysis of the relationship between player performance and economic cost.

---

## **Database Structure**

### **players**

Contains basic information about players.

- **`id`:** player identifier  
- **`first_name`:** first name  
- **`last_name`:** last name  

---

### **teams**

Contains information about teams.

- **`id`:** team identifier  
- **`name`:** team name  

---

### **performances**

Stores player performance data by team and year.

- **`player_id`:** player identifier  
- **`team_id`:** team identifier  
- **`year`:** year  
- **`H`:** number of hits  
- **`HR`:** number of home runs  

---

### **salaries**
Contains salary information for players.

- **`player_id`:** player identifier  
- **`team_id`:** team identifier  
- **`year`:** year  
- **`salary`:** salary in USD  

---

## **Key Relationships**

- A player can have multiple performances  
- Each performance is associated with a team  
- A player can receive multiple salaries depending on team and year  
- Teams pay salaries to players  

In [ ]:
import pandas as pd  
import sqlite3
conn = sqlite3.connect("moneyball.db")

def show_table(name):
    query = f"SELECT*FROM {name} LIMIT 5;"
    return pd.read_sql(query,conn)

In [3]:
show_table("players")

,id,first_name,last_name,bats,throws,weight,height,debut,final_game,birth_year,birth_month,birth_day,birth_city,birth_state,birth_country
0,2,Hank,Aaron,R,R,180,72,1954-04-13,1976-10-03,1934,2,5,Mobile,AL,USA
1,3,Tommie,Aaron,R,R,190,75,1962-04-10,1971-09-26,1939,8,5,Mobile,AL,USA
2,4,Don,Aase,R,R,190,75,1977-07-26,1990-10-03,1954,9,8,Orange,CA,USA
3,5,Andy,Abad,L,L,184,73,2001-09-10,2006-04-13,1972,8,25,Palm Beach,FL,USA
4,7,John,Abadie,R,R,192,72,1875-04-26,1875-06-10,1850,11,4,Philadelphia,PA,USA


In [4]:
show_table("teams")

,id,year,name,park
0,1,1884,Altoona Mountain City,None
1,2,2001,Anaheim Angels,Edison International Field
2,3,2001,Arizona Diamondbacks,Bank One Ballpark
3,4,2001,Atlanta Braves,Turner Field
4,5,1874,Baltimore Canaries,Newington Park


In [5]:
show_table("performances")

,id,player_id,team_id,year,G,AB,H,2B,3B,HR,RBI,SB
0,1,23,134,1871,1,4,0,0,0,0,0,0
1,2,115,112,1871,25,118,32,6,0,0,13,8
2,3,264,45,1871,29,137,40,4,5,0,19,3
3,4,268,142,1871,27,133,44,10,2,2,27,1
4,5,449,112,1871,25,120,39,11,3,0,16,6


In [6]:
show_table("salaries")

,id,player_id,team_id,year,salary
0,1,863,4,1985,870000
1,2,1171,4,1985,550000
2,3,1272,4,1985,545000
3,4,2758,4,1985,633333
4,5,3096,4,1985,625000


### **Subqueries**

Teams where Satchel Paige played 

### **Approach**

**1.** Get the player id 

**2.** Get the multiple teams' id where the player recorded a performance

**3.** Get the associated teams' names

In [ ]:
query = """ 

SELECT "name" FROM "teams" WHERE "id" IN (
    SELECT "team_id" FROM "performances" WHERE "player_id" = (
        SELECT "id" FROM "players" 
        WHERE "first_name" = 'Satchel' AND "last_name" = 'Paige'
    )
);

"""

pd.read_sql(query,conn)

,name
0,Cleveland Indians
1,Kansas City Athletics
2,St. Louis Browns


### **AGGREGATIONS AND GROUP BY**

Top 5 teams with the highest total number of hits in 2001

### **Approach**

In the same year, many performances are associated to the same team, by different players.

**1.** Join 'performances' and 'teams' tables on teams' id

**2.** Filter data by the year 2001

**3.** Sum 'hits' grouping by teams' id

In [ ]:
query = """ 

SELECT "name", SUM("H") AS "Total Hits" FROM "teams"
JOIN "performances" ON "teams"."id" = "performances"."team_id"
WHERE "performances"."year" = 2001 
GROUP BY "performances"."team_id"
ORDER BY "Total Hits" DESC
LIMIT 5;

"""

pd.read_sql(query,conn)

,name,Total Hits
0,Colorado Rockies,1663
1,Seattle Mariners,1637
2,Texas Rangers,1566
3,Cleveland Indians,1559
4,Minnesota Twins,1514


### **NESTED SUBQUERIES**

Total salary in 2001 of the player with the most home runs that year

### **Approach**

**1.** Sum all HR per player in 2001

**2.** Identify the player with the most HR

**3.** Sum all salaries received by that player in 2001

In [ ]:
query = """ 

SELECT SUM("salary") FROM "salaries" WHERE "player_id" = (
  SELECT "player_id" FROM (
    SELECT "player_id", SUM("HR") FROM "performances"
    WHERE "year" = 2001
    GROUP BY "player_id"
    ORDER BY SUM("HR") DESC
    LIMIT 1
  )
)
AND "year" = 2001;

"""

pd.read_sql(query,conn)

,"SUM(""salary"")"
0,10300000


### **Joins**

Top 10 least expensive players per hit in 2001

### **Approach**

We don´t join 'performances' and 'salaries' tables by teams' id since we are assuming that each player is associated with just one salary and performance in 2001 (one team)

In [ ]:
query = """ 

SELECT "first_name","last_name", "salary"/"H" AS "Dollars per hit"
FROM "salaries" JOIN "players" ON 
"salaries"."player_id" = "players"."id"
JOIN "performances" ON 
"performances"."player_id" = "salaries"."player_id"
AND "performances"."year" = "salaries"."year"
WHERE "salaries"."year" = 2001 AND "H" > 0
ORDER BY "Dollars per hit" ASC , "first_name" ASC, "last_name" ASC
LIMIT 10;

"""

pd.read_sql(query,conn)

,first_name,last_name,Dollars per hit
0,Albert,Pujols,1030
1,Juan,Pierre,1064
2,Jimmy,Rollins,1111
3,David,Eckstein,1204
4,Doug,Mientkiewicz,1295
5,Luis,Rivas,1333
6,Terrence,Long,1352
7,Paul,Lo Duca,1564
8,Torii,Hunter,1564
9,Aramis,Ramirez,1574
